# SO101 pick&place 本番学習（Colab / GPU用）

**事前準備（Colabを開く前に）**
1. Macで作った `so101_tape_pickplace_dataset.zip`（`outputs/`フォルダの中にあります）を、Googleドライブの `MyDrive/so101/` フォルダにアップロードしてください（無ければ作成）
2. Colab上部メニュー「ランタイム」→「ランタイムのタイプを変更」→ハードウェアアクセラレータを **GPU** にする（有料のコンピューティングユニットがあれば A100 か V100、無ければ T4）
3. 上から順にセルを実行（Shift+Enter）してください

In [ ]:
# GPUがちゃんと割り当てられているか確認（何か表示されればOK、"command not found"ならGPUが付いていない）
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# アップロードしたzipを解凍（パスが違う場合はここを修正してください）
DRIVE_ZIP = "/content/drive/MyDrive/so101/so101_tape_pickplace_dataset.zip"

!mkdir -p /content/dataset
!unzip -q "$DRIVE_ZIP" -d /content/dataset
!ls /content/dataset

In [ ]:
# lerobotをMacと同じバージョンでインストール（[dataset]で映像処理用のavも一緒に入れる）
!pip install -q "lerobot[dataset,training]==0.6.0" accelerate

In [ ]:
# 解凍したデータセットの実際のフォルダ名を自動検出
import glob

candidates = glob.glob("/content/dataset/so101_tape_pickplace_*")
assert len(candidates) == 1, f"データセットフォルダが1つに絞れません: {candidates}"
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)

In [ ]:
# 本番学習（100,000ステップ）。GPUなら数時間以内に終わる想定
# output_dirを直接Googleドライブにすることで、5000ステップごとのチェックポイントが
# 都度ドライブに残るようにしている（途中で切断されても進捗が消えないように）
!lerobot-train \
  --dataset.repo_id=local/so101_tape_pickplace \
  --dataset.root="$DATASET_ROOT" \
  --policy.type=act \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --output_dir=/content/drive/MyDrive/so101/train_output/act_tape_pickplace \
  --job_name=act_tape_pickplace \
  --steps=100000 \
  --batch_size=8 \
  --save_freq=5000 \
  --log_freq=50 \
  --wandb.enable=false

In [ ]:
# output_dirが最初からGoogleドライブなので、ここでのコピー作業は不要。
# 完了確認だけ行う（チェックポイント一覧が表示されればOK）
!ls -la /content/drive/MyDrive/so101/train_output/act_tape_pickplace/checkpoints